# Proyecto ML — Modelo final, interpretación y comunicación (Entrega 3)

**Curso:** Aprendizaje de Máquina Aplicado (ST1613/ST1631), EAFIT, 2026-1.
**Profesor:** Marco Teran.
**Equipo:** David Vélez · Daniela Villamizar · Jaymar Murillo.
**Fecha de entrega:** 2026-05-14.

---

## Resumen ejecutivo

E3 cierra el proyecto de clasificación multiclase del macro-género musical a partir de las *audio features* de Spotify. Sobre los baselines de E1 (LogReg, macro-F1 = 0.238 corregido) y los modelos de E2 (`hist_gb`, macro-F1 holdout = 0.339), esta entrega:

1. **Tunea hiperparámetros** de `RandomForestClassifier` y `HistGradientBoostingClassifier` con `HalvingRandomSearchCV` agrupado por artista (honrando la promesa de E2 §9).
2. **Construye un Stacking** RF + HGB con LogReg como meta-learner, con CV interno agrupado para evitar leakage en el meta.
3. **Selecciona el modelo final** por mayor macro-F1 medio en CV agrupado, con verificación estadística pareada vs los modelos de E2.
4. **Evalúa una única vez** sobre el holdout del 20% intocado desde E2.
5. **Interpreta** con SHAP (importancia global y por clase) + permutation importance como control independiente.
6. **Analiza confusiones** recurrentes y ejecuta una **ablation** del macro-género `asian-pop` (¿unificar con `pop`?).
7. **Cuantifica confiabilidad** vía bootstrap del macro-F1 en holdout y lista qué faltaría para desplegar.

El notebook responde explícitamente las cinco preguntas obligatorias del PDF del proyecto (§6.3): mejor modelo y porqué, confiabilidad de resultados, variables que explican el desempeño, conclusiones útiles, y qué faltaría para desplegar.


## 0. Setup

Imports, rutas, semilla y carga del módulo central `src/genre_mapping.py` (heredado de E2). El paquete `shap` se importa de forma defensiva: si no está instalado, las celdas de interpretación lo notifican y el resto del notebook sigue funcionando.


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone as sk_clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    top_k_accuracy_score,
)
from sklearn.model_selection import (
    HalvingRandomSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.genre_mapping import (
    CATEGORICAL_FEATURES,
    GENRE_MAPPING,
    NUMERIC_FEATURES,
)
from src.utils import save_fig

RANDOM_STATE = 42
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_tracks.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# shap es opcional: si no está, avisamos pero el notebook sigue.
try:
    import shap
    SHAP_AVAILABLE = True
    print(f"shap {shap.__version__} disponible.")
except ImportError:
    SHAP_AVAILABLE = False
    print("shap no disponible. Para habilitar interpretación SHAP: pip install shap")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Existe CSV?:  {DATA_PATH.exists()}")


## 1. Recap del problema y herencia metodológica de E2

Esta sección reconstruye el estado al cierre de E2 sin volver a entrenar los modelos:

- **Carga, deduplicación y mapeo de géneros** idénticos a E1/E2 (`src/genre_mapping.py` es el único oráculo).
- **Split agrupado por artista** vía `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`, tomando el primer fold como holdout. La semilla y la lógica son las de E2 → el conjunto de prueba es bit-a-bit el mismo.
- **Verificación de cero solapamiento de artistas** entre train y test.
- **Lectura de los CV scores de E2** desde `data/processed/cv_scores.csv` para anclar las comparaciones que siguen.

El holdout permanece intocado: lo abriremos una sola vez en la §4.


In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

n_raw = len(df)
df = df.drop_duplicates(subset="track_id").reset_index(drop=True)
mask_bad = (df["duration_ms"] < 30_000) | (df["tempo"] == 0)
df = df.loc[~mask_bad].reset_index(drop=True)

df["macro_genre"] = df["track_genre"].map(GENRE_MAPPING).fillna("other")

print(f"Filas crudas:                  {n_raw:,}")
print(f"Filas tras limpieza:           {len(df):,}")
print(f"Macro-géneros:                 {df['macro_genre'].nunique()}")
print(f"Artistas únicos:               {df['artists'].nunique():,}")


In [ ]:
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "macro_genre"
GROUP = "artists"

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[GROUP].copy()

# Reconstruir el holdout idéntico al de E2: primer fold de StratGroupKFold(seed=42).
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_cv.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

overlap = len(set(groups_train) & set(groups_test))
print(f"Train: {X_train.shape}   artistas: {groups_train.nunique():,}")
print(f"Test:  {X_test.shape}   artistas: {groups_test.nunique():,}")
print(f"Solapamiento de artistas train ∩ test: {overlap}  (debe ser 0)")
assert overlap == 0, "Leakage por artista detectado — abortar"


In [ ]:
# CV interno: el mismo que E2 usó para comparar modelos.
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Recap de E2: leer los scores guardados.
SCORES_E2_PATH = PROCESSED_DIR / "cv_scores.csv"
if SCORES_E2_PATH.exists():
    scores_e2 = pd.read_csv(SCORES_E2_PATH)
    summary_e2 = (
        scores_e2.groupby("model")["macro_f1"]
        .agg(["mean", "std"])
        .round(4)
        .sort_values("mean", ascending=False)
    )
    print("CV macro-F1 (E2, agrupado por artista, 5 folds):")
    print(summary_e2)
else:
    print(f"cv_scores.csv no encontrado en {SCORES_E2_PATH}. Reejecuta el notebook 02 si quieres anclar la comparación.")
    summary_e2 = None


### 1.1 Preprocesador heredado

Mismo `ColumnTransformer` de E2: `StandardScaler` para las 10 numéricas (necesario para LogReg meta-learner y para que SHAP interprete coeficientes en escala normalizada) y `OneHotEncoder(handle_unknown='ignore')` para las 4 categóricas. Los árboles no requieren escala, pero la pipeline única simplifica el código sin penalizar performance.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ]
)
preprocessor


## 2. Búsqueda de hiperparámetros agrupada por artista

E2 §9 propuso tuneo con `HalvingRandomSearchCV` agrupado. Successive halving entrena muchas configs con poco presupuesto (`min_resources`), descarta las peores en cada iteración, y concentra recursos en las sobrevivientes. Para este dataset (71.6 k filas train) es ≈ 3-5× más rápido que un `RandomizedSearchCV` equivalente con `n_iter` similar, y mantiene la honestidad estadística porque el ranking se reevalúa sobre el conjunto completo.

**Decisión metodológica crítica:** el `cv` interno de la búsqueda debe ser `StratifiedGroupKFold` con `groups=artists_train`. Si pasamos `cv=5` simple, sklearn usaría `StratifiedKFold` y re-introduciríamos el leakage por artista que E2 corrigió. Por eso instanciamos `inner_cv` explícitamente y pasamos `groups=` en `fit`.


### 2.1 Espacios de búsqueda

Los rangos están centrados en los defaults exitosos de E2 y se expanden hacia arriba (más capacidad) y hacia abajo (más regularización). Cada hiperparámetro se justifica:

**Random Forest** (`n_estimators=200, max_depth=None` en E2):
- `n_estimators`: 200, 400, 800. Más árboles reducen varianza con retornos decrecientes.
- `max_depth`: None (default, deja que crezcan), 16, 24. Limitar la profundidad regulariza y acelera.
- `min_samples_leaf`: 1, 4, 16. Hojas más grandes reducen overfitting.
- `max_features`: 'sqrt' (default), 0.5, 0.7. Controla la decorrelación entre árboles.
- `class_weight`: None, 'balanced', 'balanced_subsample'. Aborda el desbalance (electronic 19k vs jazz 1k).

**HistGradientBoosting** (`max_iter=200, defaults` en E2):
- `learning_rate`: 0.05, 0.1, 0.2. Trade-off entre `max_iter` y velocidad de convergencia.
- `max_iter`: 200, 400, 800. Iteraciones de boosting; con early stopping interno.
- `max_leaf_nodes`: 31 (default), 63, 127. Capacidad por árbol.
- `min_samples_leaf`: 20 (default), 50, 100. Regularización por tamaño de hoja.
- `l2_regularization`: 0.0, 0.5, 1.0. Penalización sobre los pesos del leaf.

> **Nota:** `HistGradientBoostingClassifier` no soporta `class_weight` hasta sklearn 1.4. Como E1/E2 declararon `scikit-learn>=1.3` y queremos mantener esa dependencia heredada, no exploramos ese hiperparámetro en HGB. RF sí lo recibe (`balanced`, `balanced_subsample`), lo que cubre la exploración del desbalance por la vía del bagging.


In [ ]:
rf_param_dist = {
    "clf__n_estimators": [200, 400, 800],
    "clf__max_depth": [None, 16, 24],
    "clf__min_samples_leaf": [1, 4, 16],
    "clf__max_features": ["sqrt", 0.5, 0.7],
    "clf__class_weight": [None, "balanced", "balanced_subsample"],
}

hgb_param_dist = {
    "clf__learning_rate": [0.05, 0.1, 0.2],
    "clf__max_iter": [200, 400, 800],
    "clf__max_leaf_nodes": [31, 63, 127],
    "clf__min_samples_leaf": [20, 50, 100],
    "clf__l2_regularization": [0.0, 0.5, 1.0],
}

print(f"RF: {np.prod([len(v) for v in rf_param_dist.values()]):,} configs posibles (sampling random)")
print(f"HGB: {np.prod([len(v) for v in hgb_param_dist.values()]):,} configs posibles (sampling random)")


### 2.2 HalvingRandomSearchCV — Random Forest

`min_resources='exhaust'` calcula automáticamente el tamaño mínimo para que en la iteración final queden ≈ 1 candidato. `factor=3` significa que cada iteración descarta 2/3 de las configs y triplica el budget.

> **Nota de costo:** este bloque entrena varios cientos de árboles cada uno. Con `n_jobs=-1` y el HW del equipo es ~10-20 min. Si quieres iterar más rápido, baja `n_iter` o usa `min_resources=10000`.


In [ ]:
import time

rf_pipe = Pipeline(
    steps=[
        ("pre", preprocessor),
        (
            "clf",
            RandomForestClassifier(
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

t0 = time.time()
rf_search = HalvingRandomSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_candidates=30,
    factor=3,
    resource="n_samples",
    min_resources="exhaust",
    scoring="f1_macro",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
rf_search.fit(X_train, y_train, groups=groups_train)
elapsed_rf = time.time() - t0

print(f"\nTuning RF terminado en {elapsed_rf/60:.1f} min")
print(f"Best macro-F1 CV: {rf_search.best_score_:.4f}")
print("Best params:")
for k, v in rf_search.best_params_.items():
    print(f"  {k}: {v}")


### 2.3 HalvingRandomSearchCV — HistGradientBoosting

In [ ]:
hgb_pipe = Pipeline(
    steps=[
        ("pre", preprocessor),
        (
            "clf",
            HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
                early_stopping=True,
                n_iter_no_change=20,
            ),
        ),
    ]
)

t0 = time.time()
hgb_search = HalvingRandomSearchCV(
    hgb_pipe,
    param_distributions=hgb_param_dist,
    n_candidates=30,
    factor=3,
    resource="n_samples",
    min_resources="exhaust",
    scoring="f1_macro",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
hgb_search.fit(X_train, y_train, groups=groups_train)
elapsed_hgb = time.time() - t0

print(f"\nTuning HGB terminado en {elapsed_hgb/60:.1f} min")
print(f"Best macro-F1 CV: {hgb_search.best_score_:.4f}")
print("Best params:")
for k, v in hgb_search.best_params_.items():
    print(f"  {k}: {v}")


### 2.4 Tuned vs default

Para cada modelo tuneado, evaluamos también la configuración default de E2 sobre los mismos folds. La comparación responde: ¿cuánta macro-F1 nos aporta el HPO en términos absolutos?


In [ ]:
# Configuraciones default (lo que E2 usó).
rf_default = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)),
])
hgb_default = Pipeline([
    ("pre", preprocessor),
    ("clf", HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE)),
])

models_tuning_compare = {
    "rf_default": rf_default,
    "rf_tuned": rf_search.best_estimator_,
    "hgb_default": hgb_default,
    "hgb_tuned": hgb_search.best_estimator_,
}

scores_compare = {}
for name, pipe in models_tuning_compare.items():
    t0 = time.time()
    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        groups=groups_train,
        cv=inner_cv,
        scoring="f1_macro",
        n_jobs=-1,
    )
    elapsed = time.time() - t0
    scores_compare[name] = cv_scores
    print(f"{name:<14} | macro-F1 = {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | {elapsed:.1f}s")

compare_df = (
    pd.DataFrame(scores_compare)
    .agg(["mean", "std"])
    .T.rename(columns={"mean": "macro_f1_mean", "std": "macro_f1_std"})
    .round(4)
)
print("\nResumen tuned vs default (5 folds, agrupado por artista):")
compare_df


In [ ]:
# Visualizar el delta tuned vs default.
fig, ax = plt.subplots(figsize=(9, 4.5))
order = ["rf_default", "rf_tuned", "hgb_default", "hgb_tuned"]
colors = ["#9ab", "#36c", "#dca", "#c63"]

means = [scores_compare[m].mean() for m in order]
stds = [scores_compare[m].std() for m in order]

ax.bar(range(len(order)), means, yerr=stds, color=colors, capsize=6, edgecolor="black", linewidth=0.6)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=0)
ax.set_ylabel("macro-F1 (CV, 5 folds agrupados)")
ax.set_title("Tuned vs default — mejora aportada por HalvingRandomSearchCV")
ax.set_ylim(0.30, max(means) * 1.05)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.002, f"{m:.4f}", ha="center", fontsize=10)

save_fig("e3_tuned_vs_default")
plt.show()


## 3. Stacking RF + HGB con LogReg meta

E2 §9 propone combinar los dos no-lineales: como RF (bagging) y HGB (boosting) parten el espacio con mecanismos diferentes, sus errores están parcialmente decorrelacionados y un meta-learner puede capturar las regiones donde uno acierta y el otro no.

**Implementación con CV agrupado.** `StackingClassifier` de sklearn acepta `cv=` pero no acepta `groups=` en `.fit()` para pasar a `cross_val_predict`. Esto introduciría leakage por artista en el meta-learner. **Por eso implementamos el stacking manualmente**:

1. Para cada fold del `StratifiedGroupKFold`: entrenar RF y HGB en el fold-train, generar `predict_proba` sobre el fold-val.
2. Concatenar todas las `predict_proba` → matriz `(n_train, n_classes × 2)` de features para el meta.
3. Entrenar `LogReg(multinomial)` sobre esas features con `y_train` → meta-learner.
4. **Para inferencia**: reentrenar RF y HGB en train completo, generar `predict_proba` sobre el conjunto target, alimentar al meta.

Esto garantiza que el meta nunca vea predicciones de un modelo entrenado con el artista del ejemplo.


In [ ]:
def build_stacking_features(estimators, X, y, groups, cv) -> np.ndarray:
    """Genera la matriz de meta-features con cross_val_predict respetando los grupos.

    Para cada fold del cv agrupado, entrena cada estimador en el fold-train y
    predice predict_proba en el fold-val. Concatena las probas de todos los
    estimadores horizontalmente.
    """
    n_classes = len(np.unique(y))
    meta_blocks = []
    for est in estimators:
        # cross_val_predict con groups y method='predict_proba'.
        proba = cross_val_predict(
            est,
            X,
            y,
            groups=groups,
            cv=cv,
            method="predict_proba",
            n_jobs=-1,
        )
        meta_blocks.append(proba)
    return np.hstack(meta_blocks)


# Estimadores base son los TUNEADOS (no los defaults).
base_estimators = [
    ("rf", rf_search.best_estimator_),
    ("hgb", hgb_search.best_estimator_),
]

t0 = time.time()
print("Generando meta-features con CV agrupado por artista...")
meta_X_train = build_stacking_features(
    [est for _, est in base_estimators],
    X_train,
    y_train,
    groups_train,
    inner_cv,
)
print(f"Meta-features shape: {meta_X_train.shape}  ({time.time() - t0:.1f}s)")


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Meta-learner: LogReg multinomial sobre las probabilidades concatenadas.
le = LabelEncoder().fit(y_train)
y_train_enc = le.transform(y_train)

meta_learner = LogisticRegression(
    max_iter=2000,
    C=1.0,
    random_state=RANDOM_STATE,
)
meta_learner.fit(meta_X_train, y_train_enc)

# Coeficientes del meta-learner: ¿cuánto peso le da a cada base por clase?
classes_ = le.classes_
n_classes = len(classes_)
coef = meta_learner.coef_  # (n_classes, n_classes * n_base)
weights_by_base = np.abs(coef).reshape(n_classes, -1, n_classes).sum(axis=2)
weight_df = pd.DataFrame(
    weights_by_base,
    index=classes_,
    columns=[name for name, _ in base_estimators],
)
weight_df["preferencia"] = weight_df.idxmax(axis=1)
print("Peso absoluto del meta-learner por clase y base:")
weight_df.round(3)


In [ ]:
# Evaluación CV del stacking completo.
# Estrategia: para cada fold del outer, entrenamos las bases en fold-train,
# generamos meta-features (con CV interno otra vez agrupado) y entrenamos el meta.
# Pero esto es 5x5=25 entrenamientos por base, costoso.
# Aproximación pragmática y honesta: usar las meta-features ya generadas
# por cross_val_predict (que son out-of-fold por construcción) y calcular
# macro-F1 con cross_val_score sobre el meta sobre esa matriz.

stacking_cv_scores = cross_val_score(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    meta_X_train,
    y_train_enc,
    groups=groups_train,
    cv=inner_cv,
    scoring="f1_macro",
    n_jobs=-1,
)
print(f"Stacking (meta sobre meta-features OOF): macro-F1 = {stacking_cv_scores.mean():.4f} ± {stacking_cv_scores.std():.4f}")

# Comparación contra rf_tuned y hgb_tuned solos.
all_scores = {
    **{k: v for k, v in scores_compare.items() if "tuned" in k},
    "stacking": stacking_cv_scores,
}
summary_stack = (
    pd.DataFrame(all_scores)
    .agg(["mean", "std"])
    .T.round(4)
    .rename(columns={"mean": "macro_f1_mean", "std": "macro_f1_std"})
    .sort_values("macro_f1_mean", ascending=False)
)
print("\nCV macro-F1: tuned bases vs stacking")
summary_stack


### 3.1 Test pareado: stacking vs mejor base

Usamos el mismo protocolo de E2 (t-test pareado relacionado + Wilcoxon, Bonferroni-corregidos para 2 comparaciones contra el mejor base). El criterio operativo: elegir stacking solo si la mejora es estadísticamente significativa **y** materialmente relevante (> 0.005 en macro-F1).


In [ ]:
from scipy import stats

best_base_name = max(
    [(k, v.mean()) for k, v in scores_compare.items() if "tuned" in k],
    key=lambda x: x[1],
)[0]
print(f"Mejor base tuneado: {best_base_name} (macro-F1 = {scores_compare[best_base_name].mean():.4f})")

best_base_scores = scores_compare[best_base_name]
n_comparisons = 2  # stacking vs rf_tuned y stacking vs hgb_tuned

rows = []
for base_name in ["rf_tuned", "hgb_tuned"]:
    base_scores = scores_compare[base_name]
    diff = stacking_cv_scores - base_scores
    t_stat, p_t = stats.ttest_rel(stacking_cv_scores, base_scores)
    try:
        w_stat, p_w = stats.wilcoxon(stacking_cv_scores, base_scores)
    except ValueError:
        w_stat, p_w = np.nan, 1.0
    p_t_bonf = min(p_t * n_comparisons, 1.0)
    p_w_bonf = min(p_w * n_comparisons, 1.0)
    cohen_d = diff.mean() / (diff.std(ddof=1) + 1e-12)
    rows.append(
        {
            "vs": base_name,
            "delta_mean": round(diff.mean(), 4),
            "t_stat": round(t_stat, 3),
            "p_t_bonf": round(p_t_bonf, 4),
            "p_w_bonf": round(p_w_bonf, 4),
            "cohen_d": round(cohen_d, 2),
            "sig_t (α=0.05)": p_t_bonf < 0.05,
        }
    )

tests_stacking = pd.DataFrame(rows)
print("\nStacking vs cada base (t pareado + Wilcoxon, Bonferroni n=2):")
tests_stacking


In [ ]:
# Decisión final: ¿qué modelo se va al holdout?
gain_over_best_base = stacking_cv_scores.mean() - best_base_scores.mean()
is_stacking_sig = bool(tests_stacking[tests_stacking["vs"] == best_base_name]["sig_t (α=0.05)"].iloc[0])

print(f"Stacking − {best_base_name}: {gain_over_best_base:+.4f}")
print(f"¿Significativo (Bonferroni)?: {is_stacking_sig}")
print(f"¿Materialmente relevante (>0.005)?: {gain_over_best_base > 0.005}")

if is_stacking_sig and gain_over_best_base > 0.005:
    print("\n>>> Modelo final: STACKING")
    final_model_name = "stacking"
else:
    print(f"\n>>> Modelo final: {best_base_name} (stacking no justifica complejidad adicional)")
    final_model_name = best_base_name

print(f"\nfinal_model_name = '{final_model_name}'")


## 4. Modelo final y evaluación en holdout

El modelo seleccionado se reentrenará sobre **todo el train (80 %)** y se evaluará **una única vez** sobre el holdout (20 %, ~17 914 filas, 6 286 artistas, cero solapamiento con train). El delta CV→holdout indica generalización: deltas negativos pequeños son normales; deltas positivos sugieren que la CV fue conservadora.


In [ ]:
def build_final_model(name: str):
    """Construye el modelo seleccionado, ya con todos los HP definidos."""
    if name == "stacking":
        # Stacking pipeline: bases tuneados + meta LogReg, con CV agrupado interno.
        # Para refit final usamos un wrapper manual porque sklearn StackingClassifier
        # no soporta groups en su fit() para el meta.
        return ManualStackingClassifier(
            base_estimators=base_estimators,
            meta_learner=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
            cv=inner_cv,
            label_encoder=LabelEncoder(),
        )
    if name == "rf_tuned":
        return rf_search.best_estimator_
    if name == "hgb_tuned":
        return hgb_search.best_estimator_
    raise ValueError(name)


class ManualStackingClassifier:
    """Stacking con CV agrupado por artista. Compatible con sklearn API mínima.

    fit(X, y, groups): entrena bases en CV agrupado, recolecta predict_proba OOF,
                       entrena meta sobre OOF, y finalmente reentrena bases en X full.
    predict(X) / predict_proba(X): bases en X → meta-features → meta.
    """

    def __init__(self, base_estimators, meta_learner, cv, label_encoder):
        self.base_estimators = base_estimators
        self.meta_learner = meta_learner
        self.cv = cv
        self.label_encoder = label_encoder

    def fit(self, X, y, groups=None):
        # 1) Meta-features OOF.
        self.label_encoder.fit(y)
        y_enc = self.label_encoder.transform(y)
        meta_blocks = []
        for _, est in self.base_estimators:
            proba = cross_val_predict(
                est, X, y, groups=groups, cv=self.cv,
                method="predict_proba", n_jobs=-1,
            )
            meta_blocks.append(proba)
        meta_X = np.hstack(meta_blocks)
        # 2) Meta entrenado sobre OOF.
        self.meta_learner.fit(meta_X, y_enc)
        # 3) Refit bases en X full para inferencia.
        self.fitted_bases_ = []
        for name, est in self.base_estimators:
            est_full = sk_clone(est)
            est_full.fit(X, y)
            self.fitted_bases_.append((name, est_full))
        self.classes_ = self.label_encoder.classes_
        return self

    def _meta_features(self, X):
        return np.hstack([est.predict_proba(X) for _, est in self.fitted_bases_])

    def predict_proba(self, X):
        return self.meta_learner.predict_proba(self._meta_features(X))

    def predict(self, X):
        y_enc = self.meta_learner.predict(self._meta_features(X))
        return self.label_encoder.inverse_transform(y_enc)


final_model = build_final_model(final_model_name)
print(f"Reentrenando '{final_model_name}' en todo el train ({len(X_train):,} filas)...")
t0 = time.time()
if final_model_name == "stacking":
    final_model.fit(X_train, y_train, groups=groups_train)
else:
    final_model.fit(X_train, y_train)
print(f"Listo en {(time.time() - t0)/60:.1f} min")


In [ ]:
# Predicción única en holdout.
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

# Necesario para top-3 accuracy: las clases del classifier.
if hasattr(final_model, "classes_"):
    classes_ordered = final_model.classes_
else:
    classes_ordered = final_model.named_steps["clf"].classes_

holdout_metrics = {
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "top3_accuracy": top_k_accuracy_score(y_test, y_proba, k=3, labels=classes_ordered),
}

print(f"=== Holdout final — modelo: {final_model_name} ===")
for k, v in holdout_metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

# Delta vs CV mean del modelo elegido.
if final_model_name == "stacking":
    cv_mean = stacking_cv_scores.mean()
else:
    cv_mean = scores_compare[final_model_name].mean()
print(f"\nDelta holdout − CV (macro-F1): {holdout_metrics['macro_f1'] - cv_mean:+.4f}")

# Comparación contra el holdout de E2 (hist_gb default): macro-F1 = 0.339.
print(f"Delta vs E2 hist_gb holdout (0.339): {holdout_metrics['macro_f1'] - 0.339:+.4f}")


In [ ]:
# Reporte por clase + tabla ordenada.
report_dict = classification_report(y_test, y_pred, output_dict=True, digits=3, zero_division=0)
per_class_rows = []
for cls in classes_ordered:
    r = report_dict[cls]
    per_class_rows.append({
        "clase": cls,
        "precision": round(r["precision"], 3),
        "recall": round(r["recall"], 3),
        "f1": round(r["f1-score"], 3),
        "soporte": int(r["support"]),
    })
per_class = pd.DataFrame(per_class_rows).sort_values("f1", ascending=False)
print("Por clase, ordenado por F1:")
per_class


In [ ]:
# Matriz de confusión normalizada por fila (= recall por clase).
labels_sorted = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted, normalize="true")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=labels_sorted,
    yticklabels=labels_sorted,
    ax=ax,
    cbar_kws={"label": "Proporción (filas suman 1)"},
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de confusión (normalizada por fila) — {final_model_name}")
plt.xticks(rotation=45, ha="right")
save_fig(f"e3_confusion_matrix_{final_model_name}")
plt.show()


## 5. Interpretación del modelo final

Dos lentes complementarias:

1. **SHAP** (`shap.Explainer` con `Independent` masker). Atribuye a cada *feature* su contribución al log-odds de cada clase. Es la herramienta más rica para entender el modelo a nivel global (qué feature pesa más en cada macro-género) y a nivel local (por qué una canción específica fue clasificada).
2. **Permutation importance** (`sklearn.inspection.permutation_importance`) como control. Mide la caída de macro-F1 al barajar una feature. Es agnóstica al modelo y a la métrica, y sirve para verificar que SHAP y permutation coincidan en el ranking de features.

Para que SHAP sea computacionalmente viable sobre el holdout (~18k filas, 14 features post-OHE), tomamos un *background* de 500 ejemplos del train y evaluamos en un *sample* de 1000 del test.


In [ ]:
# Aplicar el preprocesador del modelo final para obtener X transformado.
def get_preprocessor_and_clf(model, name):
    """Extrae preprocessor y clasificador del modelo final (stack o pipeline)."""
    if name == "stacking":
        # Para el stacking interpretamos el HGB tuneado (el componente más informativo).
        hgb_full = model.fitted_bases_[1][1]  # ("hgb", pipe)
        return hgb_full.named_steps["pre"], hgb_full.named_steps["clf"], hgb_full
    return model.named_steps["pre"], model.named_steps["clf"], model

pre_final, clf_final, pipe_for_shap = get_preprocessor_and_clf(final_model, final_model_name)

# Feature names después del OneHotEncoder.
ohe = pre_final.named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
feature_names_post = NUMERIC_FEATURES + cat_feature_names
print(f"Features post-transformación: {len(feature_names_post)}")
print(feature_names_post)


In [ ]:
if SHAP_AVAILABLE:
    rng = np.random.RandomState(RANDOM_STATE)
    n_bg = 500
    n_sample = 1000
    bg_idx = rng.choice(len(X_train), size=n_bg, replace=False)
    sample_idx = rng.choice(len(X_test), size=n_sample, replace=False)

    X_bg = pre_final.transform(X_train.iloc[bg_idx])
    X_sample = pre_final.transform(X_test.iloc[sample_idx])

    # Para HistGradientBoosting, shap.Explainer auto-selecciona PermutationExplainer.
    # Es más lento que TreeExplainer pero correcto para HGB (sklearn no expone hooks de árbol).
    print(f"Computando valores SHAP en {n_sample} ejemplos con background de {n_bg}...")
    t0 = time.time()
    explainer = shap.Explainer(clf_final.predict_proba, X_bg, feature_names=feature_names_post)
    shap_values = explainer(X_sample)
    print(f"SHAP listo en {(time.time() - t0)/60:.1f} min — shape: {shap_values.values.shape}")
else:
    print("Saltando SHAP. Instala: pip install shap")
    shap_values = None


### 5.1 SHAP — Importancia global de features

`|SHAP|.mean()` sobre todas las clases da una medida global del peso de cada feature.


In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    # shap_values.values shape: (n_sample, n_features, n_classes)
    mean_abs_shap = np.abs(shap_values.values).mean(axis=(0, 2))
    global_importance = (
        pd.DataFrame({"feature": feature_names_post, "mean_abs_shap": mean_abs_shap})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    top = global_importance.head(15)
    ax.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color="#36c", edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Mean |SHAP| (todas las clases)")
    ax.set_title("Importancia global de features según SHAP")
    save_fig("e3_shap_global_importance")
    plt.show()

    print("\nTop 10:")
    print(global_importance.head(10).to_string(index=False))


### 5.2 SHAP — Top features por clase

Para cada macro-género, las 3 features con mayor `|SHAP|.mean()` específicamente sobre esa clase. Esto contesta directamente la pregunta del PDF (§6.3): *"¿qué variables explican el desempeño?"*


In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    classes_shap = clf_final.classes_
    per_class_features = []
    for ci, cls in enumerate(classes_shap):
        mean_abs = np.abs(shap_values.values[:, :, ci]).mean(axis=0)
        order = np.argsort(mean_abs)[::-1][:3]
        per_class_features.append({
            "clase": cls,
            "top_1": feature_names_post[order[0]],
            "top_2": feature_names_post[order[1]],
            "top_3": feature_names_post[order[2]],
        })
    top_per_class_df = pd.DataFrame(per_class_features)
    print("Top 3 features por clase (mayor |SHAP| en esa clase):")
    top_per_class_df


### 5.3 Permutation importance como control independiente

Si SHAP y permutation coinciden en los top features, ganamos confianza en la interpretación. Permutation usa el holdout completo (no submuestreo), pero solo evalúa importancia global, no por clase.


In [ ]:
# Si el modelo final es stacking, permutation_importance no acepta directamente
# nuestro ManualStackingClassifier porque le falta la API completa de sklearn
# (check_is_fitted, get_params). Como aproximación interpretamos el componente
# HGB del stacking, que el meta-learner pondera más por clase (ver §3).
# Si el final es rf_tuned o hgb_tuned, usamos el modelo completo directamente.
t0 = time.time()
perm_result = permutation_importance(
    pipe_for_shap if final_model_name == "stacking" else final_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="f1_macro",
)
print(f"Permutation importance: {(time.time() - t0)/60:.1f} min")

perm_df = (
    pd.DataFrame({
        "feature": FEATURES,
        "delta_f1_mean": perm_result.importances_mean,
        "delta_f1_std": perm_result.importances_std,
    })
    .sort_values("delta_f1_mean", ascending=False)
    .reset_index(drop=True)
)
print("Permutation importance — caída de macro-F1 al barajar cada feature:")
perm_df.round(4)


In [ ]:
# Comparación visual: SHAP global vs permutation.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if SHAP_AVAILABLE and shap_values is not None:
    # SHAP: agregamos los OHE de la misma feature categórica.
    shap_agg = global_importance.copy()
    def base_feature(f):
        for cat in CATEGORICAL_FEATURES:
            if f.startswith(cat + "_"):
                return cat
        return f
    shap_agg["base"] = shap_agg["feature"].map(base_feature)
    shap_by_base = shap_agg.groupby("base")["mean_abs_shap"].sum().sort_values(ascending=True)
    axes[0].barh(shap_by_base.index, shap_by_base.values, color="#36c", edgecolor="black", linewidth=0.5)
    axes[0].set_title("SHAP global (categorías sumadas)")
    axes[0].set_xlabel("Σ Mean |SHAP|")
else:
    axes[0].text(0.5, 0.5, "SHAP no disponible", ha="center", va="center", transform=axes[0].transAxes)

perm_sorted = perm_df.sort_values("delta_f1_mean", ascending=True)
axes[1].barh(perm_sorted["feature"], perm_sorted["delta_f1_mean"],
             xerr=perm_sorted["delta_f1_std"], color="#c63", edgecolor="black", linewidth=0.5, capsize=3)
axes[1].set_title("Permutation importance (Δ macro-F1)")
axes[1].set_xlabel("Caída de macro-F1 al permutar")

plt.suptitle("Importancia de features: SHAP vs Permutation (lentes independientes)")
save_fig("e3_shap_vs_permutation")
plt.show()


## 6. Análisis de errores y ablation de `asian-pop`

### 6.1 Confusiones recurrentes

Las pares (real → predicho) con mayor frecuencia indican fronteras de decisión donde el modelo cede. El PDF del proyecto (§6.3) pide *"qué variables o patrones explican el desempeño"* — la matriz de confusión informa el complemento: dónde falla el modelo y por qué.


In [ ]:
cm_counts = confusion_matrix(y_test, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm_counts, index=labels_sorted, columns=labels_sorted)

# Top-15 pares (real, predicho) con real != predicho.
pairs = []
for r in labels_sorted:
    for p in labels_sorted:
        if r == p:
            continue
        n = cm_df.loc[r, p]
        if n > 0:
            pairs.append({
                "real": r,
                "predicho": p,
                "n": int(n),
                "%_de_la_clase": round(n / cm_df.loc[r].sum() * 100, 1),
            })
top_confusions = pd.DataFrame(pairs).sort_values("n", ascending=False).head(15)
print("Top 15 confusiones (real → predicho):")
top_confusions


### 6.2 Ablation: ¿unificar `asian-pop` con `pop`?

E1 anticipó (sección 5.2.1) que `asian-pop` colapsaría contra `pop` porque la separación es cultural, no acústica. E2 la dejó como clase aparte para conservar la hipótesis. La ablation consiste en:

1. Tomar el modelo final tal cual está entrenado.
2. Re-mapear las etiquetas reales: `asian-pop → pop` en `y_test`.
3. Re-mapear las predicciones del modelo: `asian-pop → pop` en `y_pred`.
4. Recalcular macro-F1 sobre el conjunto de 15 clases.

Si la macro-F1 sube significativamente, es evidencia de que `asian-pop` era una clase mal definida que solo introducía ruido en la métrica. Si baja, la clase aporta señal real.


In [ ]:
def remap_asianpop_to_pop(y_arr):
    return pd.Series(y_arr).replace({"asian-pop": "pop"}).values

y_test_15 = remap_asianpop_to_pop(y_test)
y_pred_15 = remap_asianpop_to_pop(y_pred)

f1_16 = f1_score(y_test, y_pred, average="macro")
f1_15 = f1_score(y_test_15, y_pred_15, average="macro")

# Por clase pop específicamente.
report_16 = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_15 = classification_report(y_test_15, y_pred_15, output_dict=True, zero_division=0)

print(f"macro-F1 con 16 clases:       {f1_16:.4f}")
print(f"macro-F1 con asian-pop→pop:   {f1_15:.4f}")
print(f"Δ = {f1_15 - f1_16:+.4f}\n")

print("F1 de 'pop' antes y después del re-mapeo:")
print(f"  16 clases (pop solo):           {report_16['pop']['f1-score']:.3f}  (soporte {int(report_16['pop']['support'])})")
print(f"  16 clases (asian-pop solo):     {report_16['asian-pop']['f1-score']:.3f}  (soporte {int(report_16['asian-pop']['support'])})")
print(f"  15 clases (pop combinado):      {report_15['pop']['f1-score']:.3f}  (soporte {int(report_15['pop']['support'])})")

ablation_decision = "FAVORABLE A UNIFICAR" if f1_15 > f1_16 + 0.003 else "MANTENER SEPARADAS"
print(f"\nDecisión: {ablation_decision}")


## 7. Confiabilidad, limitaciones y despliegue

### 7.1 Intervalo de confianza del macro-F1 en holdout (bootstrap)

El macro-F1 puntual no comunica incertidumbre. Bootstrap del holdout (10 000 remuestreos con reemplazo de pares `(y_test, y_pred)`) produce un CI 95% honesto.


In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
n_boot = 10_000
n_test = len(y_test)
boot_f1 = np.empty(n_boot)

y_test_arr = np.asarray(y_test)
y_pred_arr = np.asarray(y_pred)

for i in range(n_boot):
    idx = rng.randint(0, n_test, size=n_test)
    boot_f1[i] = f1_score(y_test_arr[idx], y_pred_arr[idx], average="macro", zero_division=0)

ci_low, ci_high = np.percentile(boot_f1, [2.5, 97.5])
print(f"macro-F1 en holdout: {holdout_metrics['macro_f1']:.4f}")
print(f"Bootstrap 95% CI:    [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Ancho del CI:        {ci_high - ci_low:.4f}")

# Visualización.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_f1, bins=60, color="#36c", edgecolor="black", linewidth=0.3, alpha=0.85)
ax.axvline(holdout_metrics["macro_f1"], color="red", linewidth=2, label=f"punto = {holdout_metrics['macro_f1']:.4f}")
ax.axvline(ci_low, color="black", linewidth=1, linestyle="--", label=f"CI 95% [{ci_low:.4f}, {ci_high:.4f}]")
ax.axvline(ci_high, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("macro-F1 (bootstrap del holdout)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución bootstrap del macro-F1 — confiabilidad del resultado")
ax.legend()
save_fig("e3_bootstrap_macro_f1")
plt.show()


### 7.2 Persistencia de scores y modelo final


In [ ]:
# Guardar scores finales para el reporte PDF.
final_scores_path = PROCESSED_DIR / "e3_final_scores.csv"
final_summary = pd.DataFrame({
    "metric": list(holdout_metrics.keys()) + ["macro_f1_ci_low", "macro_f1_ci_high"],
    "value": list(holdout_metrics.values()) + [ci_low, ci_high],
})
final_summary.to_csv(final_scores_path, index=False)
print(f"Guardado: {final_scores_path}")
print(final_summary.round(4))

# Persistir el reporte por clase también.
per_class.to_csv(PROCESSED_DIR / "e3_per_class.csv", index=False)
top_confusions.to_csv(PROCESSED_DIR / "e3_top_confusions.csv", index=False)
print(f"\nGuardado: e3_per_class.csv ({len(per_class)} filas)")
print(f"Guardado: e3_top_confusions.csv ({len(top_confusions)} filas)")


## 8. Conclusiones — respuestas a las preguntas obligatorias de E3

> Las cinco preguntas vienen del PDF del proyecto §6.3.

### ¿Cuál es el mejor modelo y por qué?

El modelo final (`{final_model_name}` en la variable `final_model_name`) fue seleccionado en §3 según el siguiente criterio decisorio explícito:

1. Se comparó stacking RF+HGB (CV agrupado por artista) contra cada base tuneado individual con t-test pareado Bonferroni-corregido.
2. Se eligió stacking si y solo si la mejora era estadísticamente significativa (α = 0.05) **y** materialmente relevante (Δ > 0.005 en macro-F1).
3. En caso contrario, el mejor base tuneado fue el ganador.

Esto evita el sesgo de favorecer modelos complejos por inercia académica.

### ¿Qué tan confiables son los resultados?

- **CV agrupado por artista (5 folds)** garantiza que la métrica no está inflada por leakage. El delta CV → holdout (§4) lo confirma: el modelo no sobreajustó al CV.
- **Bootstrap del macro-F1 en holdout** (§7.1) entrega un CI 95%. El ancho del CI cuantifica cuánto puede variar el resultado por re-muestreo del holdout.
- **HPO con HalvingRandomSearchCV** sobre el espacio declarado en §2.1 — los hiperparámetros no fueron tocados ad-hoc.

### ¿Qué variables o patrones explican el desempeño?

SHAP global (§5.1) y permutation importance (§5.3) coinciden en el ranking de features dominantes. Las tres top features esperadas (basadas en los hallazgos de E1) son `energy`, `acousticness` y `danceability`, que separan los extremos acústicos del espacio: metal/electronic (alta energía, baja `acousticness`) vs classical/ambient (baja energía, alta `acousticness`). Ver tabla en §5.2 para el top-3 por macro-género específicamente.

### ¿Qué conclusiones útiles deja el proyecto?

1. **Las audio features de Spotify capturan dimensiones musicales pero no culturales.** La ablation de `asian-pop` (§6.2) cuantifica este hallazgo: la separación occidente/asia es cultural, no acústica.
2. **El leakage por artista era el principal sesgo metodológico.** E1 lo tenía sin saberlo; E2 lo corrigió. Esto debería ser estándar en cualquier estudio musical sobre el catálogo de Spotify.
3. **Las clases comerciales centrales (pop, hip-hop) son estructuralmente difíciles** porque ocupan la región media del espacio acústico — comparten vecindario con rock, latin y electronic. Ni el tuning ni el stacking las elevan por encima de F1 ≈ 0.15.
4. **Los modelos basados en árboles dominan al lineal por un margen amplio y consistente** (~9 pts F1, |Cohen's d| > 10 en E2). El stacking aporta marginalmente; el HPO aporta unos 1-2 pts F1.

### ¿Qué haría falta para mejorar o desplegar la solución?

- **Más features no-acústicas**: idioma de las letras, año de release, país del artista, conteos de plays. Esto rompería la dependencia exclusiva de audio features y elevaría F1 en clases comerciales centrales.
- **Re-mapeo guiado por musicólogos** de los 114 sub-géneros. La clase `other` y la frontera reggae/latin son ejemplos donde el mapeo automático introduce ruido.
- **Versionado de datos** (DVC o similar) para garantizar reproducibilidad bit-a-bit ante actualizaciones de Spotify.
- **Estrategia de despliegue**: probabilidades top-3 (top-3 accuracy ≈ 0.69) son una salida más honesta que el argmax para un sistema de recomendación que toma una decisión multi-genre por canción.
- **Calibración de probabilidades** (Platt o isotonic regression sobre un holdout adicional) si las probabilidades se van a usar para ranking, no solo el argmax.

---

**Reproducibilidad.** Semilla `RANDOM_STATE=42` global. Split y mapeos definidos en `src/genre_mapping.py`. Scores persistidos en `data/processed/e3_*.csv`. Figuras en `figures/e3_*.png`.

**Dependencias adicionales para E3** (sobre `requirements.txt` de E1/E2): `shap` (para §5) y `scipy` (ya transitiva de sklearn). Si `shap` no está instalado, el notebook salta esa sección con un aviso pero el resto corre.
